# Day 7: Quantum Machine Learning

**Clemson Quantum Club** · SC Quantathon v3 Bootcamp · [clemsonquantum.com](https://clemsonquantum.com)

In this notebook:
1. Setup
2. Encoding classical data into qubits
3. Quantum kernels and the support vector machine
4. A variational quantum classifier
5. Gradients: the parameter-shift rule and training
6. Barren plateaus
7. What transfers to the hackathon
8. A kernel entry on the live machine (optional)

> Cells marked **Your turn** have a few lines for you to fill in. Cells marked **Checkpoint** contain `assert` statements: if the cell runs without an error, the answer above it is correct. Solutions are in the companion solutions notebook.

## 1. Setup

Quantum machine learning, in the form that runs today, is Day 6's variational loop pointed at data: a circuit that depends on the input $x$ and on trainable angles $\theta$, an expectation value read off it, and a classical optimizer. The two new ingredients are the **feature map**, the circuit that turns numbers into a state, and the classical machinery around it, for which this notebook uses scikit-learn. Both datasets below are synthetic and two-dimensional so that every decision boundary can be drawn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles, make_moons
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

np.set_printoptions(precision=3, suppress=True)

SHOTS = 1000
RUN_ON_HARDWARE = False     # section 8 only; False uses the recorded run
rng = np.random.default_rng(seed=3)

estimator = StatevectorEstimator()
sampler = StatevectorSampler(seed=np.random.default_rng(7))

def scale_01(X):
    return (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

X_circ, y_circ = make_circles(n_samples=120, factor=0.4, noise=0.08, random_state=3)
X_moon, y_moon = make_moons(n_samples=120, noise=0.15, random_state=3)
X_circ, X_moon = scale_01(X_circ), scale_01(X_moon)
y_circ, y_moon = 2 * y_circ - 1, 2 * y_moon - 1        # labels in {-1, +1}

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, X, y, name in [(axes[0], X_circ, y_circ, "circles"), (axes[1], X_moon, y_moon, "moons")]:
    ax.scatter(*X[y < 0].T, s=18, label="class -1"); ax.scatter(*X[y > 0].T, s=18, label="class +1")
    ax.set_title(name); ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2"); ax.set_aspect("equal")
axes[0].legend(frameon=False); plt.show()

try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(name="scqv3")
    print("IBM account 'scqv3' loaded.")
except Exception as e:
    service = None
    print("No saved IBM account:", type(e).__name__, "(fine until section 8)")

## 2. Encoding classical data into qubits

A classical data point is a list of numbers; a qubit register is a state vector. The circuit that maps one to the other is the **feature map** $U(x)$, and the state it prepares, $|\phi(x)\rangle = U(x)|0\cdots0\rangle$, is the only way the data ever enter the model. Three encodings cover most of what is used.

- **Basis encoding**: a bitstring $x \in \{0,1\}^n$ becomes the basis state $|x\rangle$. One qubit per bit, no superposition.
- **Angle encoding**: each feature $x_i \in [0, 1]$ becomes a rotation angle, $|\phi(x)\rangle = \bigotimes_i R_y(\pi x_i)|0\rangle$. One qubit per feature, a product state.
- **Amplitude encoding**: a vector of $2^n$ numbers, normalized, becomes the amplitudes of an $n$-qubit state. Exponentially compact, but preparing it costs a circuit of depth $2^n$ in general.

In [ ]:
# Basis encoding: the bitstring 101 on three qubits
qc_basis = QuantumCircuit(3)
for q, bit in enumerate(reversed("101")):
    if bit == "1":
        qc_basis.x(q)
print("basis:    ", Statevector(qc_basis).probabilities_dict())

# Angle encoding: two features as two R_y angles
def angle_map(x):
    qc = QuantumCircuit(2)
    qc.ry(np.pi * x[0], 0)
    qc.ry(np.pi * x[1], 1)
    return qc

print("angle:    ", Statevector(angle_map([0.3, 0.8])).data)

### Your turn: amplitude encoding

Encode the four numbers `x` as the amplitudes of a two-qubit state: normalize `x` to unit length into `x_norm` and build `sv = Statevector(x_norm)`. The probability of reading $k$ is then $x_k^2 / \sum_j x_j^2$. (Qiskit's `QuantumCircuit.prepare_state` builds the circuit that does this.)

In [ ]:
x = np.array([3.0, 1.0, 2.0, 4.0])
x_norm = None
sv = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert sv is not None, "fill in the cell above"
assert np.isclose(np.linalg.norm(x_norm), 1)
assert np.allclose(sv.probabilities(), x ** 2 / np.sum(x ** 2))
print("amplitudes:", sv.data, "   probabilities:", sv.probabilities())

### 2.2 Feature maps with entanglement

A product-state encoding can be simulated feature by feature, so a kernel built from it is classically easy. The feature maps used in practice add entangling gates and products of features. The **entangling map** below encodes each feature as an $R_y$ angle, entangles the pair with a CNOT, and then rotates by the product $\pi x_0 x_1$, so the state depends on the features jointly. It is the map every kernel and classifier in this notebook uses.

The **ZZ feature map** of Havlíček et al. (2019) is the best-known alternative: $H$ on every qubit, a phase $2x_i$ on each, a phase $2(\pi - x_0)(\pi - x_1)$ on the pair through two CNOTs, repeated twice. Both are built below; section 3 compares them.

In [ ]:
def entangling_map(x):
    qc = QuantumCircuit(2)
    qc.ry(np.pi * x[0], 0)
    qc.ry(np.pi * x[1], 1)
    qc.cx(0, 1)
    qc.ry(np.pi * x[0] * x[1], 1)
    return qc

def zz_map(x, reps=2):
    qc = QuantumCircuit(2)
    for _ in range(reps):
        qc.h([0, 1])
        qc.p(2 * x[0], 0)
        qc.p(2 * x[1], 1)
        qc.cx(0, 1)
        qc.p(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)
        qc.cx(0, 1)
    return qc

feature_map = entangling_map
entangling_map([0.3, 0.8]).draw("mpl", fold=-1)

In [ ]:
# Checkpoint: the map is a valid state for any input, and different inputs give different states
phi_a = Statevector(feature_map([0.3, 0.8]))
phi_b = Statevector(feature_map([0.9, 0.1]))
assert np.isclose(np.sum(np.abs(phi_a.data) ** 2), 1)
assert abs(phi_a.inner(phi_b)) ** 2 < 0.99
print("overlap of two encoded points:", round(abs(phi_a.inner(phi_b)) ** 2, 4))

## 3. Quantum kernels and the support vector machine

Many classifiers use the data only through inner products between pairs of points. A **kernel** $k(x, z)$ is such an inner product in some feature space, and a support vector machine (SVM) finds the boundary that separates the classes with the widest margin in that space, using only the kernel matrix $K_{ij} = k(x_i, x_j)$. A **quantum kernel** is the overlap of two encoded states,

$$k(x, z) = \big|\langle\phi(x)|\phi(z)\rangle\big|^2 = \big|\langle 0|U(x)^\dagger U(z)|0\rangle\big|^2,$$

which on a processor is estimated by running $U(x)^\dagger U(z)$ and counting how often all zeros comes back. On a simulator `Statevector.inner` gives it exactly.

### Your turn: the kernel matrix

Complete `kernel_matrix(A, B, fmap)` so that it returns the array $K_{ij} = |\langle\phi(a_i)|\phi(b_j)\rangle|^2$ for the rows of `A` and `B`, with $|\phi(x)\rangle$ the `Statevector` of `fmap(x)`. Build the state of every row once, then take the overlaps.

In [ ]:
def kernel_matrix(A, B, fmap=feature_map):
    K = None

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return K

In [ ]:
# Checkpoint: symmetric with ones on the diagonal, every entry between 0 and 1
K_test = kernel_matrix(X_circ[:8], X_circ[:8])
assert K_test is not None, "kernel_matrix returns None: fill in the cell above"
assert K_test.shape == (8, 8) and np.allclose(K_test, K_test.T) and np.allclose(np.diag(K_test), 1)
assert np.all(K_test >= 0) and np.all(K_test <= 1 + 1e-9)
print(np.round(K_test[:4, :4], 3))

Train an SVM on the quantum kernel with scikit-learn's `SVC(kernel="precomputed")`, which takes the training kernel matrix instead of raw features, and compare it with two classical baselines: a linear SVM and an RBF-kernel SVM on the raw features.

In [ ]:
def split(X, y, n_train=80):
    idx = rng.permutation(len(X))
    tr, te = idx[:n_train], idx[n_train:]
    return X[tr], y[tr], X[te], y[te]

datasets = {"circles": (X_circ, y_circ), "moons": (X_moon, y_moon)}
splits = {name: split(X, y) for name, (X, y) in datasets.items()}      # one fixed split per dataset, reused below

assert kernel_matrix(X_circ[:2], X_circ[:2]) is not None, "complete kernel_matrix above first"

def fit_quantum_svm(X_train, y_train, X_test, y_test):
    K_train = kernel_matrix(X_train, X_train)
    K_test = kernel_matrix(X_test, X_train)
    clf = SVC(kernel="precomputed", C=1.0).fit(K_train, y_train)
    return clf, clf.score(K_train, y_train), clf.score(K_test, y_test)

results = {}
for name in datasets:
    X_train, y_train, X_test, y_test = splits[name]
    _, acc_train, acc_test = fit_quantum_svm(X_train, y_train, X_test, y_test)
    acc_linear = SVC(kernel="linear").fit(X_train, y_train).score(X_test, y_test)
    acc_rbf = SVC(kernel="rbf").fit(X_train, y_train).score(X_test, y_test)
    results[name] = (acc_train, acc_test, acc_linear, acc_rbf)
    print(f"{name:<8} quantum kernel: train {acc_train:.0%}  test {acc_test:.0%}   |   linear SVM {acc_linear:.0%}   RBF SVM {acc_rbf:.0%}")

# Checkpoint: the quantum kernel separates the circles, which a line cannot
assert results["circles"][1] >= 0.85 and results["circles"][1] > results["circles"][2]

The same SVM with the ZZ feature map does worse on both datasets. Its pair phase $2(\pi - x_0)(\pi - x_1)$ runs from about 9 to 20 radians as the features move across $[0, 1]$, so two nearby points land on nearly orthogonal states and the kernel matrix loses the block structure the SVM needs. Scaling the inputs by a factor $\beta < 1$ before encoding, the **kernel bandwidth**, is the usual fix; the map has to be matched to the range of the data, not just made deep.

In [ ]:
assert kernel_matrix(X_circ[:2], X_circ[:2]) is not None, "complete kernel_matrix above first"
for name in datasets:
    X_train, y_train, X_test, y_test = splits[name]
    K_train, K_test = kernel_matrix(X_train, X_train, zz_map), kernel_matrix(X_test, X_train, zz_map)
    acc_zz = SVC(kernel="precomputed").fit(K_train, y_train).score(K_test, y_test)
    print(f"{name:<8} ZZ feature map: test {acc_zz:.0%}   (entangling map: {results[name][1]:.0%})")

The decision regions make the comparison concrete. Every point of a grid is encoded, its kernel against the training set is computed, and the trained SVM labels it.

In [ ]:
assert kernel_matrix(X_circ[:2], X_circ[:2]) is not None, "complete kernel_matrix above first"
def decision_regions(ax, clf, X_train, y_train, X_test, y_test, title, n=35):
    g = np.linspace(0, 1, n)
    grid = np.array([[gx, gy] for gy in g for gx in g])
    K_grid = kernel_matrix(grid, X_train)
    Z = clf.decision_function(K_grid).reshape(n, n)
    ax.contourf(g, g, Z, levels=[-10, 0, 10], colors=["#dbe9f6", "#fde5cc"])
    ax.contour(g, g, Z, levels=[0], colors="0.3", linewidths=1)
    ax.scatter(*X_train[y_train < 0].T, s=14, color="#1f77b4"); ax.scatter(*X_train[y_train > 0].T, s=14, color="#ff7f0e")
    ax.scatter(*X_test.T, s=30, facecolors="none", edgecolors="k", linewidths=0.6)
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
for ax, name in zip(axes, datasets):
    X_train, y_train, X_test, y_test = splits[name]
    clf, _, acc_test = fit_quantum_svm(X_train, y_train, X_test, y_test)
    decision_regions(ax, clf, X_train, y_train, X_test, y_test, f"{name}: quantum kernel SVM, test {acc_test:.0%}")
plt.show()

Two things to keep in mind about kernels. The RBF baseline does just as well on data like this, so the quantum kernel earns nothing here beyond a demonstration; the argument for quantum kernels rests on feature maps whose overlaps are hard to compute classically, and on data with matching structure. And as the number of qubits grows, random feature maps push all overlaps toward zero, a phenomenon called **kernel concentration**, which makes $K$ close to the identity and the SVM useless. Feature maps have to be chosen, not just made deep.

## 4. A variational quantum classifier

The second model replaces the SVM with a trainable circuit. Encode $x$ with the feature map, apply the hardware-efficient ansatz of Day 6 with angles $\theta$, and read one expectation value,

$$f(x;\theta) = \langle\phi(x)|\,W(\theta)^\dagger\, Z_0\, W(\theta)\,|\phi(x)\rangle \in [-1, 1],$$

whose sign is the predicted class. Training means choosing $\theta$ to make $f$ close to the labels, with the squared error

$$L(\theta) = \frac{1}{N}\sum_{i=1}^{N}\big(f(x_i;\theta) - y_i\big)^2$$

as the loss. Every input and every angle is a circuit parameter, so one parameterized circuit and one Estimator call evaluate $f$ on the whole dataset.

In [ ]:
x_params = ParameterVector("x", 2)
a_params = ParameterVector("a", 6)

def ansatz(params, reps=2):
    qc = QuantumCircuit(2)
    k = 0
    for layer in range(reps + 1):
        for q in range(2):
            qc.ry(params[k], q)
            k += 1
        if layer < reps:
            qc.cx(0, 1)
    return qc

model_circuit = feature_map(x_params).compose(ansatz(a_params))
observable = SparsePauliOp("IZ")                     # Z on qubit 0 (rightmost character)
print("parameters, in the order the Estimator expects:", list(model_circuit.parameters))
model_circuit.draw("mpl", fold=-1)

In [ ]:
def model(theta, X):
    values = np.array([np.concatenate([theta, x]) for x in X])      # order: a[0..5], then x[0..1]
    return np.asarray(estimator.run([(model_circuit, observable, values)]).result()[0].data.evs)

X_train, y_train, X_test, y_test = splits["circles"]
theta0 = rng.uniform(0, 2 * np.pi, 6)
f0 = model(theta0, X_train[:5])
print("f for five training points at random angles:", f0)

# Checkpoint: outputs are expectation values of Z, so they lie in [-1, 1]
assert f0.shape == (5,) and np.all(np.abs(f0) <= 1 + 1e-9)

### Your turn: the loss

Complete `loss(theta, X, y)` so that it returns the squared-error loss $L(\theta)$ above as a float, using `model`.

In [ ]:
def loss(theta, X, y):
    result = None

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return result

In [ ]:
# Checkpoint
L0 = loss(theta0, X_train, y_train)
assert L0 is not None, "loss returns None: fill in the cell above"
assert np.isclose(L0, np.mean((model(theta0, X_train) - y_train) ** 2))
assert 0 <= L0 <= 4
print(f"loss at random angles: {L0:.3f}   (a model that guesses f = 0 everywhere has loss 1)")

## 5. Gradients: the parameter-shift rule and training

Every trainable gate here is $R_y(\theta_k) = e^{-i\theta_k Y/2}$, and for any such gate the derivative of an expectation value is another expectation value, evaluated at shifted angles:

$$\frac{\partial f}{\partial\theta_k} = \frac{1}{2}\Big[f\big(\theta_k + \tfrac{\pi}{2}\big) - f\big(\theta_k - \tfrac{\pi}{2}\big)\Big].$$

This is the **parameter-shift rule**. It is exact, not a finite-difference approximation, and it works on hardware because both terms are ordinary circuit runs. Day 6 derived it; here it drives the training. The loss gradient follows by the chain rule,

$$\frac{\partial L}{\partial\theta_k} = \frac{2}{N}\sum_i \big(f(x_i;\theta) - y_i\big)\,\frac{\partial f(x_i;\theta)}{\partial\theta_k}.$$

### Your turn: the model gradient

Complete `model_gradient(theta, X)` so that it returns an array of shape `(len(X), len(theta))` whose entry `[i, k]` is $\partial f(x_i;\theta)/\partial\theta_k$ from the parameter-shift rule. Two calls to `model` per parameter.

In [ ]:
def model_gradient(theta, X):
    grad = np.zeros((len(X), len(theta)))

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return grad

In [ ]:
# Checkpoint: parameter shift agrees with a finite difference
G = model_gradient(theta0, X_train[:4])
assert G.shape == (4, 6) and np.any(G != 0), "model_gradient is not filled in"
eps = 1e-5
for k in range(6):
    e = np.zeros(6); e[k] = eps
    fd = (model(theta0 + e, X_train[:4]) - model(theta0 - e, X_train[:4])) / (2 * eps)
    assert np.allclose(G[:, k], fd, atol=1e-6), (k, G[:, k], fd)
print("parameter-shift gradient matches finite differences")

Gradient descent: step against the gradient, watch the loss fall and the accuracy rise. The learning rate and the number of steps are the only knobs.

In [ ]:
assert loss(theta0, X_train, y_train) is not None and np.any(model_gradient(theta0, X_train[:1]) != 0), "complete loss and model_gradient in sections 4 and 5 first"

def loss_gradient(theta, X, y):
    return 2 * np.mean((model(theta, X) - y)[:, None] * model_gradient(theta, X), axis=0)

def accuracy(theta, X, y):
    return np.mean(np.sign(model(theta, X)) == y)

def train(theta, X, y, steps=60, lr=0.4):
    history = []
    for step in range(steps):
        theta = theta - lr * loss_gradient(theta, X, y)
        history.append((loss(theta, X, y), accuracy(theta, X, y)))
    return theta, np.array(history)

best = None
for restart in range(3):
    th, hist = train(rng.uniform(0, 2 * np.pi, 6), X_train, y_train)
    print(f"restart {restart}: final loss {hist[-1, 0]:.3f}, train accuracy {hist[-1, 1]:.0%}")
    if best is None or hist[-1, 0] < best[1][-1, 0]:
        best = (th, hist)
theta_best, history = best

fig, ax1 = plt.subplots(figsize=(5.5, 3.2))
ax1.plot(history[:, 0], color="#1f77b4"); ax1.set_xlabel("gradient step"); ax1.set_ylabel("loss", color="#1f77b4")
ax2 = ax1.twinx(); ax2.plot(history[:, 1], color="#ff7f0e"); ax2.set_ylabel("training accuracy", color="#ff7f0e"); ax2.set_ylim(0, 1.05)
plt.title("variational classifier on the circles data"); plt.show()

acc_vqc_test = accuracy(theta_best, X_test, y_test)
print(f"test accuracy: {acc_vqc_test:.0%}")

# Checkpoint
assert history[-1, 1] >= 0.85, f"training accuracy {history[-1, 1]:.0%} is below 85%"

In [ ]:
assert loss(theta0, X_train, y_train) is not None and np.any(model_gradient(theta0, X_train[:1]) != 0), "complete loss and model_gradient in sections 4 and 5 first"
g = np.linspace(0, 1, 35)
grid = np.array([[gx, gy] for gy in g for gx in g])
Z = model(theta_best, grid).reshape(35, 35)
plt.contourf(g, g, Z, levels=[-2, 0, 2], colors=["#dbe9f6", "#fde5cc"]); plt.contour(g, g, Z, levels=[0], colors="0.3", linewidths=1)
plt.scatter(*X_train[y_train < 0].T, s=14); plt.scatter(*X_train[y_train > 0].T, s=14)
plt.gca().set_aspect("equal"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.title(f"decision regions of f(x; theta), test {acc_vqc_test:.0%}"); plt.show()

Each gradient costs $2P$ model evaluations for $P$ parameters, each of them $N$ circuits, so a step on hardware is $2PN$ circuits before shot noise is even considered. That cost, not the mathematics, is what limits variational classifiers today.

## 6. Barren plateaus

Make the ansatz wider and deeper and something goes wrong that no optimizer can fix. For random angles in a sufficiently deep circuit, the gradient of any expectation value is zero on average and its **variance shrinks exponentially with the number of qubits**, so the landscape is flat almost everywhere and the first gradient step points nowhere (McClean et al., 2018). The cell below measures that variance directly: a hardware-efficient ansatz with as many layers as qubits, a hundred random parameter settings, and the parameter-shift gradient of $\langle Z_0 Z_1\rangle$ with respect to the first angle.

In [ ]:
def wide_ansatz(n, reps):
    params = ParameterVector("t", n * (reps + 1))
    qc = QuantumCircuit(n)
    k = 0
    for layer in range(reps + 1):
        for q in range(n):
            qc.ry(params[k], q)
            k += 1
        if layer < reps:
            for q in range(n - 1):
                qc.cx(q, q + 1)
    return qc

def first_gradient(qc, obs, theta):
    shift = np.zeros_like(theta); shift[0] = np.pi / 2
    values = np.array([theta + shift, theta - shift])
    evs = estimator.run([(qc, obs, values)]).result()[0].data.evs
    return 0.5 * (evs[0] - evs[1])

def gradient_variance(n, n_samples=100):
    qc = wide_ansatz(n, reps=n)
    obs = SparsePauliOp("I" * (n - 2) + "ZZ")
    grads = [first_gradient(qc, obs, rng.uniform(0, 2 * np.pi, qc.num_parameters)) for _ in range(n_samples)]
    return float(np.var(grads))

variances = {n: gradient_variance(n) for n in (2, 3, 5, 6, 8)}
for n, v in variances.items():
    print(f"{n} qubits: gradient variance {v:.2e}")

### Your turn: one more width

On an exponential the logarithm of the variance is linear in the qubit count, so the four-qubit value should be close to the geometric mean of its neighbours, `np.sqrt(variances[3] * variances[5])`. Store that prediction in `var_4_predicted`, then compute the real value with `gradient_variance(4)` and store it in `var_4`.

In [ ]:
var_4_predicted = None
var_4 = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint: the variance keeps falling with width, and the exponential predicts the new point
assert var_4_predicted is not None and var_4 is not None, "fill in the cell above"
assert variances[5] < var_4 < variances[3], "the four-qubit variance should sit between the three- and five-qubit values"
assert 0.5 < var_4 / var_4_predicted < 2, "the computed variance should be within a factor of two of the exponential prediction"
print(f"predicted from the neighbours {var_4_predicted:.2e}   computed {var_4:.2e}")
variances[4] = var_4
ns = sorted(variances)
assert variances[8] < variances[2] / 4, "the variance should drop by well over a factor of four from 2 to 8 qubits"
plt.semilogy(ns, [variances[n] for n in ns], "o-")
plt.xlabel("qubits (and entangling layers)"); plt.ylabel("variance of one gradient component"); plt.title("a barren plateau forming"); plt.show()

The straight line on a log axis is the exponential. Ways around it are an active research area: shallow circuits, ansätze built from the problem rather than from convenience, initializing near the identity, and structured feature maps. They all amount to not searching the whole exponentially large space at once.

## 7. What transfers to the hackathon

- Fit a classical baseline first. If a linear model or an RBF SVM already scores well, a quantum model has nothing to add, and judges will ask.
- Keep the feature count at the number of qubits you can afford. Two to eight features encoded on two to eight qubits is the realistic range for a weekend; reduce dimensionality classically before encoding.
- Scale features into a fixed range before they become angles. Unscaled inputs wrap around $2\pi$ and destroy the geometry.
- A kernel matrix on hardware costs one circuit per pair, $N(N-1)/2$ circuits for $N$ points, each needing enough shots to resolve small overlaps. Simulate first; run only the final matrix, or a subset, on a processor.
- Hold out a test set and report it. Training accuracy on 80 points says nothing.
- Check for concentration and plateaus early: if every kernel entry is near zero, or gradients are $10^{-4}$, the model cannot learn and more training will not help.

## 8. A kernel entry on the live machine (optional)

One kernel entry $k(x, z)$ measured the way a processor measures it: run $U(x)$ followed by $U(z)^\dagger$ and count how often the result is `00`. About 2 seconds of QPU time. Three numbers to compare: the exact overlap from the state vector (a classical calculation), the same circuit sampled with `SHOTS` shots on the simulator, and the processor. Otherwise the recorded counts in the cell below are used.

In [ ]:
from qiskit import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

assert kernel_matrix(X_circ[:2], X_circ[:2]) is not None, "complete kernel_matrix above first"
x_a, x_b = X_circ[0], X_circ[1]
overlap_circuit = feature_map(x_a).compose(feature_map(x_b).inverse())
overlap_circuit.measure_all()
k_exact = float(kernel_matrix(x_a[None, :], x_b[None, :])[0, 0])
k_sampled = sampler.run([overlap_circuit], shots=SHOTS).result()[0].data.meas.get_counts().get("00", 0) / SHOTS
print(f"exact k(x, z) from the state vector:   {k_exact:.4f}")
print(f"simulator, {SHOTS} shots:                {k_sampled:.4f}")

job = None
if RUN_ON_HARDWARE and service is not None:
    live = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
    isa = generate_preset_pass_manager(optimization_level=3, backend=live).run(overlap_circuit)
    job = Sampler(mode=live).run([isa], shots=SHOTS)
    counts = job.result()[0].data.meas.get_counts()
    print(f"{live.name}, {SHOTS} shots:            {counts.get('00', 0) / SHOTS:.4f}   counts: {counts}")
else:
    # Recorded run: ibm_kingston (Heron r2), 1000 shots
    CACHED_HW_COUNTS = {"00": 929, "01": 65, "10": 2, "11": 4}
    print("RUN_ON_HARDWARE is False or no account: using the recorded run below.")
    print(f"recorded run, {SHOTS} shots:         {CACHED_HW_COUNTS.get('00', 0) / SHOTS:.4f}   counts: {CACHED_HW_COUNTS}")

## 9. Summary

- A feature map $U(x)$ turns data into a state; basis, angle, and amplitude encodings are the three basic choices, and entangling maps such as the ZZ feature map are what give quantum models a feature space of their own.
- A quantum kernel is the overlap $|\langle\phi(x)|\phi(z)\rangle|^2$; with `SVC(kernel="precomputed")` it drives a standard SVM. It separates circles and moons that a line cannot, and so does a classical RBF kernel.
- A variational classifier reads $f(x;\theta) = \langle Z_0\rangle$ from the feature map followed by an ansatz and trains $\theta$ on a squared-error loss.
- The parameter-shift rule gives exact gradients from two shifted circuit runs per parameter; gradient descent on the chain-rule loss gradient trains the classifier.
- In wide, deep random circuits the gradient variance falls exponentially with the number of qubits: a barren plateau. Shallow, structured circuits are the way around it.
- Classical baselines, feature scaling, held-out test sets, and the $N(N-1)/2$ circuits of a hardware kernel matrix decide whether a hackathon QML project holds up.

## Further reading

- V. Havlíček et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209 (2019), the ZZ feature map and quantum kernel SVM
- M. Schuld and N. Killoran, "Quantum machine learning in feature Hilbert spaces," Phys. Rev. Lett. 122, 040504 (2019)
- K. Mitarai, M. Negoro, M. Kitagawa, and K. Fujii, "Quantum circuit learning," Phys. Rev. A 98, 032309 (2018), the parameter-shift rule
- J. R. McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018)
- M. Cerezo et al., "Variational quantum algorithms," Nature Reviews Physics 3, 625 (2021), a review of the whole field
- [Quantum machine learning](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning) course on IBM Quantum Learning